In [0]:
# Databricks Notebook — Silver → Gold Star Schema
# ==================================================
# Dimensions (SCD2) : dim_customer, dim_product, dim_date
# Facts     (MERGE) : fact_sales, fact_marketing
# KPIs      (6)     : 3 Sales + 3 Marketing


# ─────────────────────────────────────────────────────────────────────────────
# CELL 0 — CONFIG + IMPORTS
# ─────────────────────────────────────────────────────────────────────────────

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, DoubleType,
    DateType, BooleanType
)
from delta.tables import DeltaTable

SILVER       = "saleslt.silver"
GOLD         = "saleslt.gold"
GOLD_STORAGE = "abfss://gold@storageproject12026.dfs.core.windows.net/saleslt"

print("✓ Config ready")
print(f"  Silver       : {SILVER}")
print(f"  Gold catalog : {GOLD}")
print(f"  Gold storage : {GOLD_STORAGE}")


# ─────────────────────────────────────────────────────────────────────────────
# CELL 1 — ENFORCED SCHEMAS
# ─────────────────────────────────────────────────────────────────────────────

SCHEMA_CUSTOMER = StructType([
    StructField("CustomerID",   IntegerType(), True),
    StructField("FirstName",    StringType(),  True),
    StructField("LastName",     StringType(),  True),
    StructField("EmailAddress", StringType(),  True),
    StructField("Phone",        StringType(),  True),
    StructField("CompanyName",  StringType(),  True),
    StructField("SalesPerson",  StringType(),  True),
    StructField("ModifiedDate", StringType(),  True),
])

SCHEMA_PRODUCT = StructType([
    StructField("ProductID",         IntegerType(), True),
    StructField("Name",              StringType(),  True),
    StructField("ProductNumber",     StringType(),  True),
    StructField("Color",             StringType(),  True),
    StructField("StandardCost",      DoubleType(),  True),
    StructField("ListPrice",         DoubleType(),  True),
    StructField("Size",              StringType(),  True),
    StructField("Weight",            DoubleType(),  True),
    StructField("ProductCategoryID", IntegerType(), True),
    StructField("ProductModelID",    IntegerType(), True),
    StructField("SellStartDate",     StringType(),  True),
    StructField("SellEndDate",       StringType(),  True),
    StructField("ModifiedDate",      StringType(),  True),
])

SCHEMA_ORDER_HEADER = StructType([
    StructField("SalesOrderID",    IntegerType(), True),
    StructField("OrderDate",       StringType(),  True),
    StructField("DueDate",         StringType(),  True),
    StructField("ShipDate",        StringType(),  True),
    StructField("Status",          IntegerType(), True),
    StructField("CustomerID",      IntegerType(), True),
    StructField("SubTotal",        DoubleType(),  True),
    StructField("TaxAmt",          DoubleType(),  True),
    StructField("Freight",         DoubleType(),  True),
    StructField("TotalDue",        DoubleType(),  True),
    StructField("ShipMethod",      StringType(),  True),
    StructField("OnlineOrderFlag", IntegerType(), True),
])

SCHEMA_ORDER_DETAIL = StructType([
    StructField("SalesOrderID",       IntegerType(), True),
    StructField("SalesOrderDetailID", IntegerType(), True),
    StructField("OrderQty",           IntegerType(), True),
    StructField("ProductID",          IntegerType(), True),
    StructField("UnitPrice",          DoubleType(),  True),
    StructField("UnitPriceDiscount",  DoubleType(),  True),
    StructField("LineTotal",          DoubleType(),  True),
])

SCHEMA_LEADS = StructType([
    StructField("lead_id",           StringType(),  True),
    StructField("campaign_id",       StringType(),  True),
    StructField("first_name",        StringType(),  True),
    StructField("last_name",         StringType(),  True),
    StructField("country",           StringType(),  True),
    StructField("lead_status",       StringType(),  True),
    StructField("lead_score",        IntegerType(), True),
    StructField("interest_level",    StringType(),  True),
    StructField("estimated_value",   DoubleType(),  True),
    StructField("created_date",      StringType(),  True),
    StructField("last_contacted",    StringType(),  True),
    StructField("contact_info",      StringType(),  True),
    StructField("company_info",      StringType(),  True),
    StructField("engagement",        StringType(),  True),
    StructField("touchpoints",       StringType(),  True),
    StructField("product_interests", StringType(),  True),
    StructField("tags",              StringType(),  True),
])

print("✓ CELL 1 — Schemas defined")


# ─────────────────────────────────────────────────────────────────────────────
# CELL 2 — LOAD FROM SILVER WITH ENFORCED SCHEMA + BAD ROW FLAGGING
# ─────────────────────────────────────────────────────────────────────────────

def load_silver(table, schema, required_cols):
    df = spark.table(f"{SILVER}.{table}")
    for field in schema.fields:
        if field.name in df.columns:
            df = df.withColumn(field.name, F.col(field.name).cast(field.dataType))
    bad_condition = F.lit(False)
    for col in required_cols:
        if col in df.columns:
            bad_condition = bad_condition | F.col(col).isNull()
    df = df.withColumn("_dq_bad_row", bad_condition)
    total     = df.count()
    bad_rows  = df.filter(F.col("_dq_bad_row") == True).count()
    good_rows = total - bad_rows
    print(f"  ✓ {table:30s} total={total:,}  good={good_rows:,}  bad={bad_rows:,}")
    return df

REQUIRED = {
    "customer":           ["CustomerID", "FirstName", "LastName", "EmailAddress"],
    "product":            ["ProductID", "Name", "ListPrice"],
    "sales_order_header": ["SalesOrderID", "CustomerID", "SubTotal", "TotalDue"],
    "sales_order_detail": ["SalesOrderID", "SalesOrderDetailID", "ProductID", "OrderQty"],
    "marketingleads":     ["lead_id", "lead_status", "lead_score"],
}

df_customer = load_silver("customer",           SCHEMA_CUSTOMER,     REQUIRED["customer"])
df_product  = load_silver("product",            SCHEMA_PRODUCT,      REQUIRED["product"])
df_header   = load_silver("sales_order_header", SCHEMA_ORDER_HEADER, REQUIRED["sales_order_header"])
df_detail   = load_silver("sales_order_detail", SCHEMA_ORDER_DETAIL, REQUIRED["sales_order_detail"])
df_leads    = load_silver("marketingleads",      SCHEMA_LEADS,        REQUIRED["marketingleads"])

print("\n✓ CELL 2 — All tables loaded")


# ─────────────────────────────────────────────────────────────────────────────
# CELL 3 — CLEAN CATALOG (drop stale metadata pointing to deleted paths)
# ─────────────────────────────────────────────────────────────────────────────

GOLD_TABLES = [
    "dim_customer",
    "dim_product",
    "dim_date",
    "fact_sales",
    "fact_marketing",
]

print("── Cleaning stale catalog entries ───────────────────────")
for t in GOLD_TABLES:
    try:
        spark.sql(f"DROP TABLE IF EXISTS {GOLD}.{t}")
        print(f"  ✓ Dropped (if existed) : {GOLD}.{t}")
    except Exception as e:
        print(f"  ✗ {t} : {e}")

print("\n✓ CELL 3 — Catalog cleaned — ready to recreate all tables")


# ─────────────────────────────────────────────────────────────────────────────
# CELL 4 — DIM_CUSTOMER  (SCD2)
# ─────────────────────────────────────────────────────────────────────────────

dim_customer_path = f"{GOLD_STORAGE}/dim_customer"

df_dim_customer = df_customer.filter(F.col("_dq_bad_row") == False).select(
    F.col("CustomerID").alias("customer_key"),
    F.col("FirstName").alias("first_name"),
    F.col("LastName").alias("last_name"),
    F.concat_ws(" ", "FirstName", "LastName").alias("full_name"),
    F.col("EmailAddress").alias("email"),
    F.col("Phone").alias("phone"),
    F.col("CompanyName").alias("company_name"),
    F.col("SalesPerson").alias("sales_person"),
    F.when(F.col("CompanyName").isNotNull(), "B2B").otherwise("B2C").alias("customer_type"),
).withColumn("scd_start_date", F.current_date()) \
 .withColumn("scd_end_date",   F.lit(None).cast(DateType())) \
 .withColumn("scd_is_current", F.lit(True)) \
 .withColumn("scd_version",    F.lit(1)) \
 .withColumn("_load_ts",       F.current_timestamp())

table_exists = spark.catalog.tableExists(f"{GOLD}.dim_customer")

if not table_exists:
    df_dim_customer.write.format("delta").mode("overwrite") \
                   .option("overwriteSchema", "true").save(dim_customer_path)
    spark.sql(f"CREATE TABLE IF NOT EXISTS {GOLD}.dim_customer USING DELTA LOCATION '{dim_customer_path}'")
    n = spark.sql(f"SELECT COUNT(*) AS n FROM {GOLD}.dim_customer").collect()[0]["n"]
    print(f"  ✓ dim_customer created — {n:,} rows")
else:
    dt = DeltaTable.forPath(spark, dim_customer_path)
    dt.alias("target").merge(
        df_dim_customer.alias("source"),
        "target.customer_key = source.customer_key AND target.scd_is_current = true"
    ).whenMatchedUpdate(
        condition="target.full_name != source.full_name OR target.email != source.email",
        set={"scd_end_date": "current_date()", "scd_is_current": "false"}
    ).whenNotMatchedInsertAll().execute()
    print(f"  ✓ dim_customer SCD2 merge complete")


# ─────────────────────────────────────────────────────────────────────────────
# CELL 5 — DIM_PRODUCT  (SCD2)
# ─────────────────────────────────────────────────────────────────────────────

dim_product_path = f"{GOLD_STORAGE}/dim_product"

df_dim_product = df_product.filter(F.col("_dq_bad_row") == False).select(
    F.col("ProductID").alias("product_key"),
    F.col("Name").alias("product_name"),
    F.col("ProductNumber").alias("product_number"),
    F.col("Color").alias("color"),
    F.col("Size").alias("size"),
    F.col("Weight").alias("weight"),
    F.col("StandardCost").alias("standard_cost"),
    F.col("ListPrice").alias("list_price"),
    F.round(F.col("ListPrice") - F.col("StandardCost"), 2).alias("gross_margin"),
    F.when(F.col("ListPrice") < 100,  "Budget")
     .when(F.col("ListPrice") < 500,  "Mid-Range")
     .when(F.col("ListPrice") < 2000, "Premium")
     .otherwise("Luxury").alias("price_range"),
    F.col("ProductCategoryID").alias("product_category_id"),
    F.col("SellStartDate").alias("sell_start_date"),
    F.col("SellEndDate").alias("sell_end_date"),
    F.when(F.col("SellEndDate").isNull(), "Active").otherwise("Discontinued").alias("product_status"),
).withColumn("scd_start_date", F.current_date()) \
 .withColumn("scd_end_date",   F.lit(None).cast(DateType())) \
 .withColumn("scd_is_current", F.lit(True)) \
 .withColumn("scd_version",    F.lit(1)) \
 .withColumn("_load_ts",       F.current_timestamp())

table_exists = spark.catalog.tableExists(f"{GOLD}.dim_product")

if not table_exists:
    df_dim_product.write.format("delta").mode("overwrite") \
                  .option("overwriteSchema", "true").save(dim_product_path)
    spark.sql(f"CREATE TABLE IF NOT EXISTS {GOLD}.dim_product USING DELTA LOCATION '{dim_product_path}'")
    n = spark.sql(f"SELECT COUNT(*) AS n FROM {GOLD}.dim_product").collect()[0]["n"]
    print(f"  ✓ dim_product created — {n:,} rows")
else:
    dt = DeltaTable.forPath(spark, dim_product_path)
    dt.alias("target").merge(
        df_dim_product.alias("source"),
        "target.product_key = source.product_key AND target.scd_is_current = true"
    ).whenMatchedUpdate(
        condition="target.list_price != source.list_price OR target.product_status != source.product_status",
        set={"scd_end_date": "current_date()", "scd_is_current": "false"}
    ).whenNotMatchedInsertAll().execute()
    print(f"  ✓ dim_product SCD2 merge complete")


# ─────────────────────────────────────────────────────────────────────────────
# CELL 6 — DIM_DATE  (static)
# ─────────────────────────────────────────────────────────────────────────────

dim_date_path = f"{GOLD_STORAGE}/dim_date"

df_dim_date = spark.sql("""
    WITH date_spine AS (
        SELECT EXPLODE(SEQUENCE(
            TO_DATE('2020-01-01'),
            TO_DATE('2030-12-31'),
            INTERVAL 1 DAY
        )) AS date_key
    )
    SELECT
        date_key,
        DATE_FORMAT(date_key, 'yyyyMMdd')             AS date_id,
        YEAR(date_key)                                AS year,
        QUARTER(date_key)                             AS quarter,
        MONTH(date_key)                               AS month_number,
        DATE_FORMAT(date_key, 'MMMM')                 AS month_name,
        DATE_FORMAT(date_key, 'MMM yyyy')             AS month_year,
        WEEKOFYEAR(date_key)                          AS week_of_year,
        DAYOFMONTH(date_key)                          AS day_of_month,
        DATE_FORMAT(date_key, 'EEEE')                 AS day_name,
        CASE WHEN DAYOFWEEK(date_key) IN (1,7)
             THEN true ELSE false END                 AS is_weekend,
        CONCAT(YEAR(date_key),'-Q',QUARTER(date_key)) AS quarter_label
    FROM date_spine
""")

table_exists = spark.catalog.tableExists(f"{GOLD}.dim_date")

if not table_exists:
    df_dim_date.write.format("delta").mode("overwrite") \
               .option("overwriteSchema", "true").save(dim_date_path)
    spark.sql(f"CREATE TABLE IF NOT EXISTS {GOLD}.dim_date USING DELTA LOCATION '{dim_date_path}'")
    n = spark.sql(f"SELECT COUNT(*) AS n FROM {GOLD}.dim_date").collect()[0]["n"]
    print(f"  ✓ dim_date created — {n:,} rows")
else:
    print(f"  ✓ dim_date already exists — skipping")


# ─────────────────────────────────────────────────────────────────────────────
# CELL 7 — FACT_SALES  (MERGE)
# ─────────────────────────────────────────────────────────────────────────────

fact_sales_path = f"{GOLD_STORAGE}/fact_sales"

df_fact_sales = df_detail.filter(F.col("_dq_bad_row") == False).join(
    df_header.filter(F.col("_dq_bad_row") == False),
    "SalesOrderID", "inner"
).select(
    F.col("SalesOrderDetailID").alias("sales_order_detail_key"),
    F.col("SalesOrderID").alias("sales_order_id"),
    F.col("CustomerID").alias("customer_key"),
    F.col("ProductID").alias("product_key"),
    F.to_date(F.col("OrderDate")).alias("order_date_key"),
    F.col("OrderDate").alias("order_date"),
    F.col("DueDate").alias("due_date"),
    F.col("ShipDate").alias("ship_date"),
    F.col("OrderQty").alias("order_qty"),
    F.col("UnitPrice").alias("unit_price"),
    F.col("UnitPriceDiscount").alias("discount_rate"),
    F.round(F.col("UnitPrice") * F.col("UnitPriceDiscount") * F.col("OrderQty"), 2).alias("discount_amount"),
    F.col("LineTotal").alias("line_total"),
    F.col("SubTotal").alias("order_sub_total"),
    F.col("TaxAmt").alias("order_tax"),
    F.col("Freight").alias("order_freight"),
    F.col("TotalDue").alias("order_total_due"),
    F.col("OnlineOrderFlag").alias("is_online_order"),
    F.col("ShipMethod").alias("ship_method"),
    F.when(
        F.col("ShipDate").isNotNull(),
        F.datediff(F.col("ShipDate"), F.col("OrderDate"))
    ).alias("days_to_ship"),
).withColumn("_load_ts", F.current_timestamp())

table_exists = spark.catalog.tableExists(f"{GOLD}.fact_sales")

if not table_exists:
    df_fact_sales.write.format("delta").mode("overwrite") \
                 .option("overwriteSchema", "true").save(fact_sales_path)
    spark.sql(f"CREATE TABLE IF NOT EXISTS {GOLD}.fact_sales USING DELTA LOCATION '{fact_sales_path}'")
    n = spark.sql(f"SELECT COUNT(*) AS n FROM {GOLD}.fact_sales").collect()[0]["n"]
    print(f"  ✓ fact_sales created — {n:,} rows")
else:
    dt = DeltaTable.forPath(spark, fact_sales_path)
    dt.alias("target").merge(
        df_fact_sales.alias("source"),
        "target.sales_order_detail_key = source.sales_order_detail_key"
    ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
    print(f"  ✓ fact_sales merge complete")


# ─────────────────────────────────────────────────────────────────────────────
# CELL 8 — FACT_MARKETING  (MERGE + flatten nested JSON)
# ─────────────────────────────────────────────────────────────────────────────

fact_marketing_path = f"{GOLD_STORAGE}/fact_marketing"

df_fact_marketing = df_leads.filter(F.col("_dq_bad_row") == False).select(
    F.col("lead_id").alias("lead_key"),
    F.col("campaign_id"),
    F.col("first_name"),
    F.col("last_name"),
    F.col("country"),
    F.col("lead_status"),
    F.col("lead_score"),
    F.col("interest_level"),
    F.col("estimated_value"),
    F.to_date(F.col("created_date")).alias("created_date_key"),
    F.col("last_contacted"),
    F.get_json_object(F.col("contact_info"), "$.email").alias("email"),
    F.get_json_object(F.col("contact_info"), "$.phone").alias("phone"),
    F.get_json_object(F.col("contact_info"), "$.linkedin").alias("linkedin"),
    F.get_json_object(F.col("company_info"), "$.name").alias("company_name"),
    F.get_json_object(F.col("company_info"), "$.industry").alias("industry"),
    F.get_json_object(F.col("company_info"), "$.size").alias("company_size"),
    F.get_json_object(F.col("company_info"), "$.city").alias("city"),
    F.get_json_object(F.col("engagement"), "$.email_opens").cast("INT").alias("email_opens"),
    F.get_json_object(F.col("engagement"), "$.email_clicks").cast("INT").alias("email_clicks"),
    F.get_json_object(F.col("engagement"), "$.website_visits").cast("INT").alias("website_visits"),
    F.get_json_object(F.col("engagement"), "$.demo_requested").cast("BOOLEAN").alias("demo_requested"),
    F.get_json_object(F.col("engagement"), "$.webinar_attended").cast("BOOLEAN").alias("webinar_attended"),
    F.get_json_object(F.col("engagement"), "$.total_score").cast("INT").alias("engagement_score"),
    F.get_json_object(F.col("engagement"), "$.score_category").alias("engagement_category"),
    F.col("tags"),
    F.when(F.col("lead_score") >= 70, "Hot")
     .when(F.col("lead_score") >= 40, "Warm")
     .otherwise("Cold").alias("lead_quality"),
    F.when(F.col("lead_status") == "Converted", True)
     .otherwise(False).alias("is_converted"),
).withColumn("_load_ts", F.current_timestamp())

table_exists = spark.catalog.tableExists(f"{GOLD}.fact_marketing")

if not table_exists:
    df_fact_marketing.write.format("delta").mode("overwrite") \
                     .option("overwriteSchema", "true").save(fact_marketing_path)
    spark.sql(f"CREATE TABLE IF NOT EXISTS {GOLD}.fact_marketing USING DELTA LOCATION '{fact_marketing_path}'")
    n = spark.sql(f"SELECT COUNT(*) AS n FROM {GOLD}.fact_marketing").collect()[0]["n"]
    print(f"  ✓ fact_marketing created — {n:,} rows")
else:
    dt = DeltaTable.forPath(spark, fact_marketing_path)
    dt.alias("target").merge(
        df_fact_marketing.alias("source"),
        "target.lead_key = source.lead_key"
    ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
    print(f"  ✓ fact_marketing merge complete")


# ─────────────────────────────────────────────────────────────────────────────
# CELL 9 — KPI : SALES (3 reports)
# ─────────────────────────────────────────────────────────────────────────────

print("=" * 65)
print("  SALES KPI 1 — Daily Sales Summary")
print("=" * 65)
spark.sql(f"""
    SELECT
        order_date_key                            AS order_date,
        COUNT(DISTINCT sales_order_id)            AS total_orders,
        SUM(order_qty)                            AS total_items_sold,
        ROUND(SUM(line_total), 2)                 AS daily_revenue,
        ROUND(AVG(line_total), 2)                 AS avg_line_value,
        ROUND(AVG(days_to_ship), 1)               AS avg_days_to_ship
    FROM {GOLD}.fact_sales
    GROUP BY order_date_key
    ORDER BY order_date DESC
    LIMIT 30
""").show(truncate=False)

print("=" * 65)
print("  SALES KPI 2 — Monthly Revenue Trend")
print("=" * 65)
spark.sql(f"""
    SELECT
        d.year,
        d.month_name,
        d.month_year,
        COUNT(DISTINCT f.sales_order_id)          AS total_orders,
        SUM(f.order_qty)                          AS total_qty,
        ROUND(SUM(f.line_total), 2)               AS monthly_revenue,
        ROUND(AVG(f.order_total_due), 2)          AS avg_order_value,
        ROUND(AVG(f.days_to_ship), 1)             AS avg_days_to_ship
    FROM      {GOLD}.fact_sales f
    JOIN      {GOLD}.dim_date   d
           ON f.order_date_key = d.date_key
    GROUP BY  d.year, d.month_number, d.month_name, d.month_year
    ORDER BY  d.year, d.month_number
""").show(truncate=False)

print("=" * 65)
print("  SALES KPI 3 — Product Performance")
print("=" * 65)
spark.sql(f"""
    SELECT
        p.product_name,
        p.price_range,
        p.product_status,
        COUNT(DISTINCT f.sales_order_id)               AS total_orders,
        SUM(f.order_qty)                               AS total_qty_sold,
        ROUND(SUM(f.line_total), 2)                    AS total_revenue,
        ROUND(AVG(f.discount_rate) * 100, 2)           AS avg_discount_pct,
        ROUND(SUM(f.order_qty) * p.gross_margin, 2)    AS est_gross_profit
    FROM      {GOLD}.fact_sales  f
    JOIN      {GOLD}.dim_product p
           ON f.product_key = p.product_key
          AND p.scd_is_current = true
    GROUP BY  p.product_name, p.price_range, p.product_status, p.gross_margin
    ORDER BY  total_revenue DESC
""").show(truncate=False)


# ─────────────────────────────────────────────────────────────────────────────
# CELL 10 — KPI : MARKETING (3 reports)
# ─────────────────────────────────────────────────────────────────────────────

print("=" * 65)
print("  MARKETING KPI 1 — Daily Lead Activity")
print("=" * 65)
spark.sql(f"""
    SELECT
        created_date_key                              AS activity_date,
        COUNT(*)                                      AS new_leads,
        ROUND(AVG(lead_score), 1)                     AS avg_lead_score,
        SUM(CASE WHEN is_converted THEN 1 ELSE 0 END) AS conversions,
        ROUND(SUM(CASE WHEN is_converted THEN 1 ELSE 0 END)
              * 100.0 / COUNT(*), 1)                  AS conversion_rate_pct,
        ROUND(SUM(estimated_value), 2)                AS pipeline_value
    FROM {GOLD}.fact_marketing
    GROUP BY created_date_key
    ORDER BY activity_date DESC
    LIMIT 30
""").show(truncate=False)

print("=" * 65)
print("  MARKETING KPI 2 — Monthly Campaign ROI")
print("=" * 65)
spark.sql(f"""
    SELECT
        d.month_year,
        m.campaign_id,
        COUNT(*)                                         AS total_leads,
        SUM(CASE WHEN m.is_converted THEN 1 ELSE 0 END) AS converted,
        ROUND(SUM(CASE WHEN m.is_converted THEN 1 ELSE 0 END)
              * 100.0 / COUNT(*), 1)                     AS conversion_rate_pct,
        ROUND(AVG(m.lead_score), 1)                      AS avg_lead_score,
        ROUND(SUM(m.estimated_value), 2)                 AS total_pipeline_value,
        ROUND(SUM(CASE WHEN m.is_converted
                       THEN m.estimated_value
                       ELSE 0 END), 2)                   AS converted_value
    FROM      {GOLD}.fact_marketing m
    JOIN      {GOLD}.dim_date       d
           ON m.created_date_key = d.date_key
    GROUP BY  d.month_year, d.month_number, m.campaign_id
    ORDER BY  d.month_number, converted_value DESC
""").show(truncate=False)

print("=" * 65)
print("  MARKETING KPI 3 — Lead Conversion Funnel")
print("=" * 65)
spark.sql(f"""
    SELECT
        lead_status,
        lead_quality,
        COUNT(*)                                      AS total_leads,
        ROUND(AVG(lead_score), 1)                     AS avg_score,
        ROUND(AVG(estimated_value), 2)                AS avg_deal_value,
        ROUND(SUM(estimated_value), 2)                AS total_pipeline,
        ROUND(COUNT(*) * 100.0
              / SUM(COUNT(*)) OVER (), 1)             AS pct_of_total
    FROM {GOLD}.fact_marketing
    GROUP BY lead_status, lead_quality
    ORDER BY avg_score DESC
""").show(truncate=False)


# ─────────────────────────────────────────────────────────────────────────────
# CELL 11 — FINAL SUMMARY
# ─────────────────────────────────────────────────────────────────────────────

ALL_TABLES = {
    "dim_customer":   "DIM  (SCD2)",
    "dim_product":    "DIM  (SCD2)",
    "dim_date":       "DIM  (static)",
    "fact_sales":     "FACT (merge)",
    "fact_marketing": "FACT (merge)",
}

print("=" * 65)
print("  GOLD LAYER SUMMARY")
print("=" * 65)
print(f"  {'TYPE':<15} {'TABLE':<25} {'ROWS':>8}")
print("-" * 65)
for table, kind in ALL_TABLES.items():
    n = spark.sql(f"SELECT COUNT(*) AS n FROM {GOLD}.{table}").collect()[0]["n"]
    print(f"  {kind:<15} {table:<25} {n:>8,}")
print("=" * 65)
print()
print("  Sales KPIs    : daily_summary | monthly_revenue | product_perf")
print("  Marketing KPIs: daily_leads   | campaign_roi    | conv_funnel")
print(f"  Storage       : {GOLD_STORAGE}")
print("=" * 65)